# Setup (run every session)

Kaggle and Colab reset between sessions, so run these cells top to bottom each time.

On Kaggle: turn on **Internet** and pick the **GPU T4 x2** accelerator in the notebook settings.

In [ ]:
import os, sys

REPO_URL = "https://github.com/shoimya/Is-the-Judge-the-Bottleneck.git"
BRANCH = "main"   # change to your working branch, e.g. "SC", to run unmerged work
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
WORK = "/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else os.getcwd()
REPO = os.path.join(WORK, "Is-the-Judge-the-Bottleneck")
print("Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local", sys.version)

In [ ]:
# 1. Clone the repo, or pull the latest if it's already here
import socket
try:
    socket.gethostbyname("github.com")
except OSError:
    raise SystemExit("No internet. On Kaggle: verify your phone number (kaggle.com > Settings), "
                     "then switch Internet on in the notebook's settings panel and run all again.")

if os.path.isdir(REPO):
    !git -C {REPO} fetch origin
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
assert os.path.isdir(REPO), "Clone failed: see the git error above."
%cd {REPO}

In [ ]:
# 2. Install pinned requirements
!pip install -q -r requirements.txt

In [ ]:
# 3. Keep runs/ somewhere that survives the session
#    Kaggle: /kaggle/working/runs (kept as notebook output when you Save Version)
#    Colab:  Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_TARGET = "/content/drive/MyDrive/Is-the-Judge-the-Bottleneck/runs"
elif IN_KAGGLE:
    RUNS_TARGET = "/kaggle/working/runs"
else:
    RUNS_TARGET = None

if RUNS_TARGET:
    os.makedirs(RUNS_TARGET, exist_ok=True)
    if not os.path.islink("runs"):
        !rm -rf runs
        os.symlink(RUNS_TARGET, "runs")
    print("runs/ ->", os.path.realpath("runs"))

In [ ]:
# 4. Setup test
!python run.py --check

# T03: build the BM25 index (run once)

Downloads HotpotQA's 2017 Wikipedia (1.55 GB), builds the index (~15 min), then deletes the dump so only the ~3 GB index is kept. Afterwards: **Save Version** (Save & Run All), then create a private Kaggle Dataset from this notebook's output folder `data/wiki/bm25_index`.

In [ ]:
!python -m pipeline.retriever build
!rm -rf data/wiki/dump
!du -sh data/wiki/bm25_index

# Use the saved index (every session after the build)

Attach the private dataset `hotpotqa-bm25-index` first (**Add Input → Your Datasets**). This links it into `data/wiki/bm25_index`, rebuilds the question sets, and runs the same Pilot-set check as on the Mac.

In [ ]:
import glob, os

index_files = glob.glob("/kaggle/input/**/bm25_index/params.index.json", recursive=True)
assert index_files, "Index not found: attach the hotpotqa-bm25-index dataset (Add Input)."
os.makedirs("data/wiki", exist_ok=True)
if not os.path.exists("data/wiki/bm25_index"):
    os.symlink(os.path.dirname(index_files[0]), "data/wiki/bm25_index")

!python -m pipeline.dataset
!python -m pipeline.retriever check
!git status --short results/question_ids   # no output = exactly the same questions as on the Mac